<a href="https://colab.research.google.com/github/JoshuaFZ/QWEN-0.6B-LORA/blob/main/LORA_OWON_qwen0_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OWON Qwen3-0.6B LoRA 训练

用于示波器语音控制 NLU：输入语音识别文本，输出紧凑 JSON 的 `intent + slots`。训练目标不直接输出 SCPI，RK3588 端由业务层根据 JSON 生成 SCPI。

## 1. 挂载 Drive 并检查数据路径

把 `LORA_train-qwen0.6B.jsonl` 和 `LORA_test-qwen0.6B.jsonl` 放在 notebook 同目录，或放在 MyDrive 根目录。

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DATA_DIR_CANDIDATES = [
    Path('/content/drive/MyDrive/VoiceControl2/training/qwen3-0.6B'),
    Path('/content/drive/MyDrive/qwen3-0.6B'),
    Path('/content/drive/MyDrive'),
    Path('/content'),
]

DATA_DIR = None
for candidate in DATA_DIR_CANDIDATES:
    if (candidate / 'LORA_train-qwen0.6B.jsonl').exists() and (candidate / 'LORA_test-qwen0.6B.jsonl').exists():
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        '没有找到 LORA_train-qwen0.6B.jsonl 和 LORA_test-qwen0.6B.jsonl。'
        '请把这两个文件上传到 notebook 同目录或 MyDrive 根目录。'
    )

TRAIN_JSONL = DATA_DIR / 'LORA_train-qwen0.6B.jsonl'
TEST_JSONL = DATA_DIR / 'LORA_test-qwen0.6B.jsonl'
OUTPUT_DIR = DATA_DIR / 'owon-qwen3-0.6b-output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR:', DATA_DIR)
print('TRAIN_JSONL:', TRAIN_JSONL)
print('TEST_JSONL:', TEST_JSONL)
print('OUTPUT_DIR:', OUTPUT_DIR)


## 2. 安装依赖

如果安装后 `import unsloth` 失败，执行 Colab 的 `Runtime -> Restart runtime`，然后从第 1 步之后重新运行。

In [ ]:
!pip uninstall -y -q unsloth unsloth_zoo
!pip install -q --no-cache-dir -U   git+https://github.com/unslothai/unsloth.git   git+https://github.com/unslothai/unsloth-zoo.git   trl peft accelerate bitsandbytes datasets


In [ ]:
import importlib.util

if importlib.util.find_spec('unsloth') is None:
    raise RuntimeError('unsloth 未安装成功。请重启 Colab Runtime 后，从挂载 Drive 的单元继续运行。')

print('✅ unsloth import check passed')


## 3. 加载 Qwen3-0.6B 并配置 LoRA

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 512  # 语音控制 JSON 输出很短，512 足够，训练更快。
dtype = None
load_in_4bit = True
model_name = 'Qwen/Qwen3-0.6B'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)

print('✅ model and LoRA adapter initialized')


## 4. 加载和格式化训练/测试数据

训练和 RK3588 端推理必须使用同一个 prompt 模板。这里使用短中文模板，减少端侧 token 开销。

In [ ]:
import json
from datasets import load_dataset

PROMPT_TEMPLATE = '''任务：解析示波器语音指令，只输出JSON。
指令：{}
输入：{}
输出：{}'''

EOS_TOKEN = tokenizer.eos_token

def normalize_output(output):
    if isinstance(output, dict):
        return json.dumps(output, ensure_ascii=False, separators=(',', ':'))
    if isinstance(output, str):
        # 训练数据里的 output 是 JSON 字符串；这里做一次规范化，避免空格差异。
        try:
            return json.dumps(json.loads(output), ensure_ascii=False, separators=(',', ':'))
        except json.JSONDecodeError:
            return output.strip()
    return str(output).strip()

def formatting_prompts_func(examples):
    texts = []
    for instruction, input_text, output in zip(examples['instruction'], examples['input'], examples['output']):
        output_str = normalize_output(output)
        texts.append(PROMPT_TEMPLATE.format(instruction, input_text, output_str) + EOS_TOKEN)
    return {'text': texts}

train_dataset_raw = load_dataset('json', data_files=str(TRAIN_JSONL), split='train')
test_dataset_raw = load_dataset('json', data_files=str(TEST_JSONL), split='train')

train_dataset = train_dataset_raw.map(formatting_prompts_func, batched=True, remove_columns=train_dataset_raw.column_names)

print('train rows:', len(train_dataset_raw))
print('test rows:', len(test_dataset_raw))
print()
print('--- formatted sample ---')
print(train_dataset[0]['text'])


## 5. 训练

203 条训练数据先跑 5 轮，避免小数据过拟合。根据独立测试集结果再考虑提高到 8-10 轮。

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field='text',
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=15,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=3407,
        output_dir=str(OUTPUT_DIR / 'trainer_outputs'),
        save_strategy='no',
        report_to='none',
    ),
)

trainer_stats = trainer.train()
print(trainer_stats)


## 6. 独立测试集评估

只 decode 新生成 token，避免把 prompt 混入结果；使用确定性解码，贴近 RK3588 端控制场景。

In [ ]:
import json
from collections import Counter
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

def build_prompt(instruction, input_text):
    return PROMPT_TEMPLATE.format(instruction, input_text, '')

def generate_response(instruction, input_text, max_new_tokens=96):
    prompt = build_prompt(instruction, input_text)
    inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    gen_tokens = outputs[0][inputs.input_ids.shape[-1]:]
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

def parse_json_maybe(text):
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    # 兜底：截取第一个完整 JSON 对象，用于评估定位问题；部署端仍应要求纯 JSON。
    start = text.find('{')
    if start == -1:
        return None
    depth = 0
    in_string = False
    escape = False
    for i, ch in enumerate(text[start:], start):
        if in_string:
            if escape:
                escape = False
            elif ch == '\\':
                escape = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == '{':
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(text[start:i + 1])
                    except Exception:
                        return None
    return None

results = []
for example in test_dataset_raw:
    expected = json.loads(normalize_output(example['output']))
    generated_text = generate_response(example['instruction'], example['input'])
    generated = parse_json_maybe(generated_text)
    results.append({
        'input': example['input'],
        'expected': expected,
        'generated_text': generated_text,
        'generated': generated,
        'json_valid': generated is not None,
        'exact_match': generated == expected,
        'intent_match': generated is not None and generated.get('intent') == expected.get('intent'),
        'slots_match': generated is not None and generated.get('slots') == expected.get('slots'),
    })

n = len(results)
summary = {
    'total': n,
    'json_valid': sum(r['json_valid'] for r in results),
    'exact_match': sum(r['exact_match'] for r in results),
    'intent_match': sum(r['intent_match'] for r in results),
    'slots_match': sum(r['slots_match'] for r in results),
}

print('--- Evaluation Summary ---')
for key, value in summary.items():
    if key == 'total':
        print(f'{key}: {value}')
    else:
        print(f'{key}: {value}/{n} = {value / n:.2%}')

print()
print('--- Mismatches (first 20) ---')
shown = 0
for idx, r in enumerate(results, 1):
    if not r['exact_match']:
        print()
        print(f"#{idx} input: {r['input']}")
        print('expected:', json.dumps(r['expected'], ensure_ascii=False, separators=(',', ':')))
        print('generated_text:', r['generated_text'])
        print('generated:', r['generated'])
        shown += 1
        if shown >= 20:
            break

intent_counts = Counter(r['expected']['intent'] for r in results)
print()
print('--- Test intent distribution ---')
for intent, count in sorted(intent_counts.items()):
    print(intent, count)


## 7. 保存 LoRA Adapter 和合并模型

同时保存 adapter 和 merged 16-bit。RK3588 端通常还需要把 merged 模型转换成 GGUF 并量化，例如 q4_k_m。

In [ ]:
ADAPTER_DIR = OUTPUT_DIR / 'owon-qwen3-0.6b-lora-adapter'
MERGED_DIR = OUTPUT_DIR / 'owon-qwen3-0.6b-merged'

model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

model.save_pretrained_merged(str(MERGED_DIR), tokenizer, save_method='merged_16bit')

print('✅ adapter saved to:', ADAPTER_DIR)
print('✅ merged model saved to:', MERGED_DIR)


In [ ]:
import shutil

archive_base = OUTPUT_DIR / 'owon-qwen3-0.6b-lora-and-merged'
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=str(OUTPUT_DIR), base_dir='.')
print('✅ archive:', archive_path)


## 8. 下载结果（可选）

In [ ]:
from google.colab import files

files.download(str(OUTPUT_DIR / 'owon-qwen3-0.6b-lora-and-merged.zip'))
